In [1]:
#!pip install pycryptodome
from Crypto.Cipher import DES
from Crypto.Random import get_random_bytes
import binascii

def pad(text):
    """
    Pad the input text to be a multiple of 8 bytes, as DES requires the data size to be a multiple of 8 bytes.
    """
    while len(text) % 8 != 0:
        text += b' '
    return text

def des_encrypt(plaintext, key):
    """
    Encrypts plaintext using DES algorithm with the given key.
    
    Parameters:
    - plaintext: The data to encrypt (bytes).
    - key: The encryption key (8 bytes).
    
    Returns:
    - ciphertext: The encrypted data (bytes).
    """
    des = DES.new(key, DES.MODE_ECB)
    padded_text = pad(plaintext)
    ciphertext = des.encrypt(padded_text)
    return ciphertext

def des_decrypt(ciphertext, key):
    """
    Decrypts ciphertext using DES algorithm with the given key.
    
    Parameters:
    - ciphertext: The data to decrypt (bytes).
    - key: The decryption key (8 bytes).
    
    Returns:
    - plaintext: The decrypted data (bytes).
    """
    des = DES.new(key, DES.MODE_ECB)
    plaintext = des.decrypt(ciphertext)
    return plaintext.rstrip(b' ')

# Example usage
key = get_random_bytes(8)  # DES key must be 8 bytes long
plaintext = b"Hello, World!"

print("Original:", plaintext)

# Encrypt
ciphertext = des_encrypt(plaintext, key)
print("Encrypted:", binascii.hexlify(ciphertext))

# Decrypt
decrypted_text = des_decrypt(ciphertext, key)
print("Decrypted:", decrypted_text)

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
    --------------------------------------- 0.0/1.8 MB 435.7 kB/s eta 0:00:04
   - -------------------------------------- 0.1/1.8 MB 375.8 kB/s eta 0:00:05
   --------- ------------------------------ 0.4/1.8 MB 2.6 MB/s eta 0:00:01
   ---------------------- ----------------- 1.0/1.8 MB 4.7 MB/s eta 0:00:01
   ---------------------------------- ----- 1.5/1.8 MB 6.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 5.9 MB/s eta 0:00:00
Original: b'Hello, World!'
Encrypted: b'218d310bdf00ef9395f86147f45ea2b1'
Decrypted: b'Hello, World!'


Pseudo-code for DES Encryption:

Input:
- plaintext: 64-bit input to be encrypted
- subkeys[16]: array of 48-bit subkeys for each of the 16 rounds

Output:
- ciphertext: 64-bit encrypted output

Variables:
- left[0..15], right[0..15]: arrays to hold the left and right halves of the data during the rounds
- temp: temporary variable used for swapping

Steps:
1. Apply the initial permutation (IP) to the plaintext
   data = IP(plaintext)

2. Split the permuted data into two equal halves
   left[0] = data[0..31]  // First 32 bits
   right[0] = data[32..63]  // Last 32 bits

3. Perform 16 rounds of processing
   for round from 1 to 16 do:
     a. Apply the round function (F) to the right half and the round's subkey
        temp = F(right[round-1], subkeys[round-1])
     
     b. XOR the output of the round function with the left half
        temp = temp XOR left[round-1]
     
     c. The left half for the next round is the right half of the current round
        left[round] = right[round-1]
     
     d. The right half for the next round is the result stored in temp
        right[round] = temp

4. After the final round, combine the halves in reverse order (right[16], left[16])
   combined_data = right[16] + left[16]

5. Apply the final permutation (FP) to the combined data
   ciphertext = FP(combined_data)

6. Return the ciphertext as the output

In [9]:
def DES_encrypt(plaintext, subkeys):
    """
    Encrypts a 64-bit plaintext using the DES algorithm.

    Parameters:
    - plaintext: 64-bit input to be encrypted.
    - subkeys[16]: Array of 48-bit subkeys for each of the 16 rounds.

    Returns:
    - ciphertext: 64-bit encrypted output.
    """

    # Initial Permutation (IP)
    permuted_data = IP(plaintext)

    # Splitting permuted data into left and right halves
    left = [0] * 17
    right = [0] * 17
    left[0], right[0] = permuted_data[:32], permuted_data[32:]

    # 16 rounds of processing
    for round in range(1, 17):
        # Applying the round function (F) to the right half and the current round's subkey
        # XOR the result with the left half
        # The left half for the next round becomes the right half of the current round
        # The right half for the next round is the result of the XOR operation
        temp = F(right[round - 1], subkeys[round - 1])
        temp = XOR(temp, left[round - 1])
        left[round], right[round] = right[round - 1], temp

    # Combining the halves in reverse order before the final permutation
    combined_data = right[16] + left[16]

    # Final Permutation (FP)
    ciphertext = FP(combined_data)

    return ciphertext

# Helper functions (assumed to be provided and implemented elsewhere):
def IP(data):
    """
    Performs the Initial Permutation (IP) on 64-bit data.
    
    Parameters:
    - data: 64-bit input as a string of 0s and 1s.
    
    Returns:
    - permuted_data: 64-bit output as a string of 0s and 1s after permutation.
    """
    # IP table defines the position of each bit in the output
    IP_table = [58, 50, 42, 34, 26, 18, 10, 2,
                60, 52, 44, 36, 28, 20, 12, 4,
                62, 54, 46, 38, 30, 22, 14, 6,
                64, 56, 48, 40, 32, 24, 16, 8,
                57, 49, 41, 33, 25, 17, 9, 1,
                59, 51, 43, 35, 27, 19, 11, 3,
                61, 53, 45, 37, 29, 21, 13, 5,
                63, 55, 47, 39, 31, 23, 15, 7]
    permuted_data = ''.join([data[IP_table[i]-1] for i in range(64)])
    return permuted_data

def F(right_half, subkey):
    """
    The round function (F) for DES, which takes a 32-bit half and a 48-bit subkey.
    
    Parameters:
    - right_half: 32-bit input as a string of 0s and 1s.
    - subkey: 48-bit subkey as a string of 0s and 1s.
    
    Returns:
    - output: 32-bit output as a string of 0s and 1s.
    """
    # Placeholder for the actual functionality of F, which involves expansion,
    # substitution, and permutation steps. This is a simplified version.
    # Normally, you would expand right_half to 48 bits, XOR with subkey,
    # apply the S-boxes, and then a final permutation.
    output = right_half  # Simplified placeholder
    return output

def FP(data):
    """
    Performs the Final Permutation (FP) on 64-bit data.
    
    Parameters:
    - data: 64-bit input as a string of 0s and 1s.
    
    Returns:
    - permuted_data: 64-bit output as a string of 0s and 1s after permutation.
    """
    # FP table is the inverse of the IP table
    FP_table = [40, 8, 48, 16, 56, 24, 64, 32,
                39, 7, 47, 15, 55, 23, 63, 31,
                38, 6, 46, 14, 54, 22, 62, 30,
                37, 5, 45, 13, 53, 21, 61, 29,
                36, 4, 44, 12, 52, 20, 60, 28,
                35, 3, 43, 11, 51, 19, 59, 27,
                34, 2, 42, 10, 50, 18, 58, 26,
                33, 1, 41, 9, 49, 17, 57, 25]
    permuted_data = ''.join([data[FP_table[i]-1] for i in range(64)])
    return permuted_data

def XOR(bits1, bits2):
    """
    Performs bitwise XOR operation on two bitstrings.
    
    Parameters:
    - bits1: First bitstring.
    - bits2: Second bitstring.
    
    Returns:
    - result: Resultant bitstring after XOR.
    """
    result = ''.join(['1' if b1 != b2 else '0' for b1, b2 in zip(bits1, bits2)])
    return result

KeyboardInterrupt: 

In [5]:
def DES_encrypt(plaintext, subkeys):
    """
    Encrypts the given plaintext using the DES algorithm.

    Parameters:
    plaintext (str): The plaintext to be encrypted.
    subkeys (list): A list of subkeys generated from the DES key.

    Returns:
    str: The ciphertext generated from the encryption process.
    """
    
    # Apply IP
    permuted_data = IP(plaintext)
    left, right = [0] * 17, [0] * 17
    left[0], right[0] = permuted_data[:32], permuted_data[32:]
    
    # 16 rounds
    for round in range(1, 17):
        temp = XOR(F(right[round - 1], subkeys[round - 1]), left[round - 1])
        left[round], right[round] = right[round - 1], temp
    
    # Combine and apply FP
    combined_data = right[16] + left[16]
    ciphertext = FP(combined_data)
    return ciphertext

def DES_decrypt(ciphertext, subkeys):
    """
    Decrypts the given ciphertext using the DES algorithm.

    Args:
        ciphertext (str): The ciphertext to be decrypted.
        subkeys (list): A list of subkeys used in the decryption process.

    Returns:
        str: The decrypted plaintext.

    """
    # Apply IP
    permuted_data = IP(ciphertext)
    left, right = [0] * 17, [0] * 17
    left[0], right[0] = permuted_data[:32], permuted_data[32:]
    
    # 16 rounds with subkeys in reverse
    for round in range(1, 17):
        temp = XOR(F(right[round - 1], subkeys[16 - round]), left[round - 1])
        left[round], right[round] = right[round - 1], temp
    
    # Combine and apply FP
    combined_data = right[16] + left[16]
    plaintext = FP(combined_data)
    return plaintext

def generate_subkeys(key):
    #assert len(key) == 64, "Key must be exactly 64 bits long"
    # Initial key permutation (PC1) to reduce 64-bit key to 56 bits
    PC1 = [57, 49, 41, 33, 25, 17, 9, 1, 58, 50, 42, 34, 26, 18,
           10, 2, 59, 51, 43, 35, 27, 19, 11, 3, 60, 52, 44, 36,
           63, 55, 47, 39, 31, 23, 15, 7, 62, 54, 46, 38, 30, 22,
           14, 6, 61, 53, 45, 37, 29, 21, 13, 5, 28, 20, 12, 4]
   
    # Subkey rotation schedule
    rotations = [1, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1]
    
    # Key compression permutation (PC2) to reduce 56-bit key to 48 bits
    PC2 = [14, 17, 11, 24, 1, 5, 3, 28, 15, 6, 21, 10,
           23, 19, 12, 4, 26, 8, 16, 7, 27, 20, 13, 2,
           41, 52, 31, 37, 47, 55, 30, 40, 51, 45, 33, 48,
           44, 49, 39, 56, 34, 53, 46, 42, 50, 36, 29, 32]
    
    # Apply PC1 to the key
    key = ''.join([key[i-1] for i in PC1])
    
    # Split the key into two halves
    left, right = key[:28], key[28:]
    
    subkeys = []
    for rotation in rotations:
        # Rotate left and right halves
        left = left[rotation:] + left[:rotation]
        right = right[rotation:] + right[:rotation]
        
        # Combine halves and apply PC2 to generate subkey
        combined_key = left + right
        subkey = ''.join([combined_key[i-1] for i in PC2])
        subkeys.append(subkey)
    
    return subkeys

# Example usage (assuming subkeys are generated and helper functions are implemented)
subkeys = generate_subkeys(key)  # Placeholder for subkey generation
plaintext = "64-bit message"
encrypted = DES_encrypt(plaintext, subkeys)
decrypted = DES_decrypt(encrypted, subkeys)
print("Original:", plaintext)
print("Encrypted:", encrypted)
print("Decrypted:", decrypted)

TypeError: 'int' object is not subscriptable

In [6]:
# Define the initial permutation (IP) table
IP = [58, 50, 42, 34, 26, 18, 10, 2,
      60, 52, 44, 36, 28, 20, 12, 4,
      62, 54, 46, 38, 30, 22, 14, 6,
      64, 56, 48, 40, 32, 24, 16, 8,
      57, 49, 41, 33, 25, 17, 9, 1,
      59, 51, 43, 35, 27, 19, 11, 3,
      61, 53, 45, 37, 29, 21, 13, 5,
      63, 55, 47, 39, 31, 23, 15, 7]

# Define the final permutation (FP) table
FP = [40, 8, 48, 16, 56, 24, 64, 32,
      39, 7, 47, 15, 55, 23, 63, 31,
      38, 6, 46, 14, 54, 22, 62, 30,
      37, 5, 45, 13, 53, 21, 61, 29,
      36, 4, 44, 12, 52, 20, 60, 28,
      35, 3, 43, 11, 51, 19, 59, 27,
      34, 2, 42, 10, 50, 18, 58, 26,
      33, 1, 41, 9, 49, 17, 57, 25]

# Simplified for demonstration purposes
def apply_permutation(data, table):
    return ''.join(data[i-1] for i in table)

def generate_subkeys(key):
    # This is a simplified version. Actual implementation should reduce 64-bit key to 56 bits using PC1,
    # split it, perform left shifts, and then apply PC2 to generate each of the 16 48-bit subkeys.
    subkeys = [""] * 16  # Placeholder for generated subkeys
    for i in range(16):
        subkeys[i] = key[48*i:48*(i+1)]  # Simplified; replace with actual subkey generation logic
    return subkeys

def DES_encrypt(plaintext, key):
    subkeys = generate_subkeys(key)
    permuted_data = apply_permutation(plaintext, IP)
    left, right = permuted_data[:32], permuted_data[32:]
    
    for round in range(16):
        # Placeholder for the round function, including expansion, XOR with subkey, substitution, and permutation
        new_right = left  # Simplified; replace with actual round function logic
        left = right
        right = new_right
    
    combined_data = right + left  # Swap back
    ciphertext = apply_permutation(combined_data, FP)
    return ciphertext

def DES_decrypt(ciphertext, key):
    subkeys = generate_subkeys(key)
    permuted_data = apply_permutation(ciphertext, IP)
    left, right = permuted_data[:32], permuted_data[32:]
    
    for round in range(16):
        # Placeholder for the round function, using subkeys in reverse order
        new_right = left  # Simplified; replace with actual round function logic
        left = right
        right = new_right
    
    combined_data = right + left  # Swap back
    plaintext = apply_permutation(combined_data, FP)
    return plaintext

# Example usage
key = "0001001100110100010101110111100110011011101111001101111111110001"  # Example 64-bit key
plaintext = "0000000100100011010001010110011110001001101010111100110111101111"  # Example 64-bit plaintext

ciphertext = DES_encrypt(plaintext, key)
decrypted_text = DES_decrypt(ciphertext, key)

print("Original:", plaintext)
print("Encrypted:", ciphertext)
print("Decrypted:", decrypted_text)

Original: 0000000100100011010001010110011110001001101010111100110111101111
Encrypted: 0000001000010011100010101001101101000110010101111100111011011111
Decrypted: 0000000100100011010001010110011110001001101010111100110111101111


In [14]:
# Define the initial permutation (IP) function
def IP(data):
    ip_table = [58, 50, 42, 34, 26, 18, 10, 2,
                60, 52, 44, 36, 28, 20, 12, 4,
                62, 54, 46, 38, 30, 22, 14, 6,
                64, 56, 48, 40, 32, 24, 16, 8,
                57, 49, 41, 33, 25, 17, 9, 1,
                59, 51, 43, 35, 27, 19, 11, 3,
                61, 53, 45, 37, 29, 21, 13, 5,
                63, 55, 47, 39, 31, 23, 15, 7]
    return ''.join(data[i-1] for i in ip_table)

# Define the final permutation (FP) function
def FP(data):
    fp_table = [40, 8, 48, 16, 56, 24, 64, 32,
                39, 7, 47, 15, 55, 23, 63, 31,
                38, 6, 46, 14, 54, 22, 62, 30,
                37, 5, 45, 13, 53, 21, 61, 29,
                36, 4, 44, 12, 52, 20, 60, 28,
                35, 3, 43, 11, 51, 19, 59, 27,
                34, 2, 42, 10, 50, 18, 58, 26,
                33, 1, 41, 9, 49, 17, 57, 25]
    return ''.join(data[i-1] for i in fp_table)

# Define the XOR function
def XOR(bits1, bits2):
    return ''.join(str(int(b1) ^ int(b2)) for b1, b2 in zip(bits1, bits2))

# Define the F function (simplified version)
def F(right, subkey):
    # This is a placeholder for the actual F function which includes expansion, S-box substitution, and permutation
    # For demonstration, we'll just return a XOR of right and subkey
    return XOR(right, subkey)

def generate_subkeys(key):
    # Simplified version; actual implementation should involve PC1, shifts, and PC2
    subkeys = [""] * 16
    for i in range(16):
        subkeys[i] = key[48*i:48*(i+1)]
    return subkeys

def DES_encrypt(plaintext, key):
    subkeys = generate_subkeys(key)
    permuted_data = IP(plaintext)
    left, right = permuted_data[:32], permuted_data[32:]
    print('here', left, right)
    
    for round in range(16):
        left, right = right, XOR(left, F(right, subkeys[round]))
        print('here1', subkeys[round], left, right)
   
    combined_data = right + left
    ciphertext = FP(combined_data)
    return ciphertext

def DES_decrypt(ciphertext, key):
    subkeys = generate_subkeys(key)
    permuted_data = IP(ciphertext)
    left, right = permuted_data[:32], permuted_data[32:]
    
    for round in range(15, -1, -1):
        new_right = XOR(left, F(right, subkeys[round]))
        left = right
        right = new_right
    
    combined_data = right + left
    plaintext = FP(combined_data)
    return plaintext

# Example usage
key = "0001001100110100010101110111100110011011101111001101111111110001"
plaintext = "0000000100100011010001010110011110001001101010111100110111101111"
print(len(key), len(plaintext))

ciphertext = DES_encrypt(plaintext, key)
decrypted_text = DES_decrypt(ciphertext, key)

print("Original:", plaintext)
print("Encrypted:", ciphertext)
print("Decrypted:", decrypted_text)

64 64
here 11001100000000001100110011111111 11110000101010101111000010101010
here1 000100110011010001010111011110011001101110111100 11110000101010101111000010101010 00101111100111100110101100101100
here1 1101111111110001 00101111100111100110101100101100 0000000011000101
here1  0000000011000101 
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   
here1   


IndexError: string index out of range

In [25]:
def IP(data):
    ip_table = [58, 50, 42, 34, 26, 18, 10, 2,
                60, 52, 44, 36, 28, 20, 12, 4,
                62, 54, 46, 38, 30, 22, 14, 6,
                64, 56, 48, 40, 32, 24, 16, 8,
                57, 49, 41, 33, 25, 17, 9, 1,
                59, 51, 43, 35, 27, 19, 11, 3,
                61, 53, 45, 37, 29, 21, 13, 5,
                63, 55, 47, 39, 31, 23, 15, 7]
    return ''.join(data[i-1] for i in ip_table)

def F(right, subkey):
    # This is a placeholder for the actual F function which includes expansion, S-box substitution, and permutation
    # For demonstration, we'll just return a XOR of right and subkey
    return XOR(right, subkey)

def FP(data):
    fp_table = [40, 8, 48, 16, 56, 24, 64, 32,
                39, 7, 47, 15, 55, 23, 63, 31,
                38, 6, 46, 14, 54, 22, 62, 30,
                37, 5, 45, 13, 53, 21, 61, 29,
                36, 4, 44, 12, 52, 20, 60, 28,
                35, 3, 43, 11, 51, 19, 59, 27,
                34, 2, 42, 10, 50, 18, 58, 26,
                33, 1, 41, 9, 49, 17, 57, 25]
    return ''.join(data[i-1] for i in fp_table)

def generate_subkeys(key):
    # This is a simplified version; actual implementation should involve PC1, shifts, and PC2
    # Assuming key is a binary string
    #subkeys = [key[i:(i+48)%56] for i in range(0, len(key), 48)]  # Split key into 48-bit parts
    import numpy as np
    np.random.seed(1)
    key = np.array(list(key))
    subkeys = [''.join(key[np.random.choice(56, 48)].tolist()) for i in range(16)]
    return subkeys[:16]  # Ensure only 16 subkeys are returned

def XOR(bits1, bits2):
    return ''.join(str(int(b1) ^ int(b2)) for b1, b2 in zip(bits1, bits2))

def DES_encrypt(plaintext, key):
    subkeys = generate_subkeys(key)
    initialPermutation = IP(plaintext)
    left, right = initialPermutation[:32], initialPermutation[32:]
    
    for round in range(16):
        new_left = right
        new_right = XOR(left, F(right, subkeys[round]))
        left, right = new_left, new_right
    
    finalPermutation = FP(right + left)  # Note the reversal of right and left
    return finalPermutation

def DES_decrypt(ciphertext, key):
    subkeys = generate_subkeys(key)
    initialPermutation = IP(ciphertext)
    left, right = initialPermutation[:32], initialPermutation[32:]
    
    for round in range(15, -1, -1):
        #print(subkeys[round])
        new_left = right
        new_right = XOR(left, F(right, subkeys[round]))
        left, right = new_left, new_right
    
    finalPermutation = FP(right + left)  # Note the reversal of right and left
    return finalPermutation

# Example usage
key = "00010011001101000101011101111001111110011011111111100101"  # Example 56-bit key
plaintext = "0000000100100011010001010110011110001001101010111100110111101111"  # Example 64-bit plaintext

ciphertext = DES_encrypt(plaintext, key)
decrypted_text = DES_decrypt(ciphertext, key)

print("Original:", plaintext)
print("Encrypted:", ciphertext)
print("Decrypted:", decrypted_text)

Original: 0000000100100011010001010110011110001001101010111100110111101111
Encrypted: 1111100111100101100010110100111010110100001111101110001000111000
Decrypted: 0000000100100011010001010110011110001001101010111100110111101111
